In [ ]:
# ===== CONFIG =====
import os
INPUT_DIR   = os.environ.get("ITDA_INPUT_DIR",  "./val_images")
OUTPUT_PATH = os.environ.get("ITDA_OUTPUT_PATH", "./submission.csv")
# ==================

# 소비기한 추출 파이프라인 (2-Pass Hybrid, CRAFT 탐지 + PP-OCRv5 인식)

```
원본 → ① EXIF 보정 → ② 1패스: 축소본 전체 OCR → 날짜 후보 + 위치
                     → ③ 2패스: 후보 영역만 원본 해상도 크롭 → 재인식
                     → ④ 선택: 가장 늦은 날짜 (= 소비기한, 운영진 확정 규칙)
                     → ⑤ 후보 없음 → 사전확률 날짜 (부분점수 정책)
```

- 비싼 고해상도 인식을 전체가 아닌 **후보 영역에만** 쓴다. 1패스는 리콜, 2패스는 정밀도.
- 9자리 이상 연속 숫자(품목보고번호·바코드)는 파싱 전에 마스킹한다.
- 탐지는 EasyOCR CRAFT, 인식은 PaddleOCR `en_PP-OCRv5_mobile_rec` (실측 133장: EasyOCR 인식기 44.4% → PP-OCRv5 60.9%). `ITDA_REC=easyocr` 로 예전 인식기로 되돌릴 수 있다.
- 모든 가중치는 `./weights` 에서 오프라인 로드한다 (`download_weights.sh` 선실행).

In [ ]:
import re, glob, time
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps
import easyocr

# torch 기본값은 물리 코어 수. 논리 코어까지 쓰면 검출기가 ~25% 빨라진다 (실측: 4→8 스레드 12.3s→8.9s).
TORCH_THREADS  = int(os.environ.get("ITDA_THREADS", 0)) or os.cpu_count() or 4   # ITDA_THREADS 는 개발용 (4코어 채점 환경 시뮬레이션)
torch.set_num_threads(TORCH_THREADS)

# ----- 파이프라인 파라미터 -----
USE_GPU        = False
WEIGHTS_DIR    = "./weights"

# 인식기 백엔드. "paddle" = PaddleOCR PP-OCRv5 영문 인식기 (기본), "easyocr" = EasyOCR english_g2 (예전 방식, 비교용).
# 실측(133장, 탐지·후처리 동일): EasyOCR 44.4% / PP-OCRv3 53.4% / PP-OCRv4 58.6% / PP-OCRv5 60.9%. 인식 호출당 200ms → 70ms.
REC_BACKEND    = os.environ.get("ITDA_REC", "paddle")
PADDLE_REC_DIR = os.path.join(WEIGHTS_DIR, "en_PP-OCRv5_mobile_rec")   # inference.json/.pdiparams/.yml, config.json (약 8MB)
PADDLE_THREADS = 4
REC_BATCH      = 8 if REC_BACKEND == "paddle" else 1   # Paddle 은 크롭 여러 개를 한 번에 인식 (실측 16개: 498ms → 321ms). EasyOCR 은 순서 보장이 없어 1개씩.

# 탐지기 백엔드. "paddle" = PaddleOCR PP-OCRv5 mobile det (기본), "craft" = EasyOCR CRAFT (예전 방식, 비교용).
# 실측(4스레드, 640px): CRAFT 2,300ms → PP-OCRv5 det 293ms. 4코어 채점 환경에서 CRAFT 는 장당 5.8s 로 2,400s 한도 초과.
# Windows 의 paddle 3.3.1 은 탐지 모델에서 oneDNN 오류(ConvertPirAttribute2RuntimeAttribute)가 나므로 enable_mkldnn=False 로 고정.
DET_BACKEND    = os.environ.get("ITDA_DET", "paddle")
PADDLE_DET_DIR = os.path.join(WEIGHTS_DIR, "PP-OCRv5_mobile_det")      # 약 5MB
os.environ.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")   # 오프라인: 모델 저장소 연결 확인 생략

# 1패스 해상도 사다리(긴 변 px). 앞 단계에서 후보가 나오면 멈추고, 없을 때만 다음 단계로 올라간다.
# 검출기(CRAFT) 비용은 픽셀 수에 비례 — 실측(4스레드): 480→1.8s, 640→2.9s, 800→4.6s, 1024→7.8s, 1280→12.2s.
# 640 에서는 영양성분표 잔글씨가 검출되지 않아 잡음이 줄고 큼직한 날짜 스탬프는 잡히지만,
# 고해상도 사진에 작게 인쇄된 날짜(3024x4032 의 라벨 등)는 놓친다 → 그런 장만 1024 로 재시도.
# 채점 예산 4.8s/장(500장/2400s). 대부분 640 에서 끝나 평균은 낮고, 어려운 장에만 예산을 더 쓴다.
PASS1_LADDER   = (640, 1024)

# 1패스 인식 예산. 인식기는 박스당 ~0.17s(실측)라 글자 많은 라벨(50박스=8.5s)에서 검출기보다 비싸다.
# 박스를 글자 높이 내림차순으로 인식하다가 연도 포함 날짜를 찾으면, 그 높이의 EARLY_STOP_RATIO 배 미만인 박스는 건너뛴다.
# (날짜 스탬프는 영양성분표 잔글씨보다 크고, 제조일자·소비기한 짝은 보통 같은 글꼴 크기.)
MAX_BOXES        = 30          # 장당 인식 박스 상한 → 최악 5초
EARLY_STOP_RATIO = 0.6

# 2패스용 원본 상한(긴 변 px). 이보다 큰 JPEG 은 디코딩 단계(draft)에서 축소한다.
# 24MP 원본을 그대로 풀면 디코딩+EXIF 회전만 2~3초. 날짜 스탬프 재인식에는 2000px 이면 충분하다.
LOAD_MAX_LONG  = 2000
PASS2_MARGIN   = 0.15          # 2패스 크롭 여백 비율 (박스 크기 대비)

# 연도 허용 범위 2017~2031 (docs/날짜_해석_규칙.md 정본. 라벨 800장 실측 최소 2017, 최대 2030).
# 상한을 넓히면 '33 01 15' 같은 잡음이 2033년으로 통과해 '가장 늦은 날짜' 규칙을 오염시킨다.
YEAR_MIN, YEAR_MAX = 2017, 2031
YY_FLIP_FROM = 2028   # 2자리 연도 3숫자에서 년/월/일로 읽은 연도가 이 값 이상이고 일/월/년도 성립하면 일/월/년 (정본 §4-3. 라벨 800장 중 2028+ 는 4건)

DEBUG = os.environ.get("ITDA_DEBUG", "") == "1"   # 개발용: 장별 OCR 라인·후보·소요시간 출력. 채점 환경에선 미설정.
MASK_DIGITS_GE = 9             # 이 길이 이상 연속 숫자는 날짜가 아님 (품목보고번호 11~14자리, 바코드 13자리)

# 인식 allowlist. 영문 월(FEB/26/21, 04-Jul-21, DEC-28-2021)과 키워드 힌트(BBD, BEST BEFORE, DD/MM/YY)를 읽으려면 영문자가 필요하다.
# 영문자를 허용하면 숫자가 O/I/S/B 로 읽힐 수 있으므로 파싱 전에 fix_confusions() 로 되돌린다. 실측 비교용으로 끌 수 있게 플래그로 둔다.
ALLOW_LETTERS = os.environ.get("ITDA_ALLOW_LETTERS", "1") == "1"
_LETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz" if ALLOW_LETTERS else ""
ALLOW_P1 = "0123456789./-: " + _LETTERS   # 1패스 (시각 구분자 ':' 포함해 시간을 시간으로 읽게 함)
ALLOW_P2 = "0123456789./- "  + _LETTERS   # 2패스

# 후보 0개(OCR 전면 실패)일 때 정책.
#  - "none" : NONE,NONE,NONE. 스펙 준수 기본값. 운영진 확정: NONE 은 정답 라벨 값으로 실제 존재한다(예: 연도 없는 우유 → NONE-10-14).
#  - "prior": 사전확률 날짜. 라벨이 실제 날짜인데 OCR 이 통째로 놓친 경우엔 기대값이 높다. 검증셋으로 비교 후 결정.
NONE_POLICY = "none"           # "none" | "prior"
PRIOR_DATE  = (2027, 1, 1)     # TODO: 검증셋 라벨 분포로 갱신

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

In [ ]:
def load_image(path, max_long=LOAD_MAX_LONG):
    """EXIF Orientation 보정 후 BGR ndarray 반환.
    - 데이터셋의 8.7%가 Orientation=6 (시계 90도). cv2.imread 는 EXIF 를 무시하므로 PIL 로 연다.
    - MPO(.jpg 확장자의 다중프레임), PNG-as-jpg 등 포맷 불일치도 PIL 이 흡수한다.
    - JPEG 은 draft 모드로 디코딩 단계에서 1/2·1/4·1/8 축소한다. 요청 크기 이상은 보장되므로 max_long 은 하한이다.
    - EasyOCR 은 cv2 관행(BGR) 입력을 가정하므로 채널 순서를 맞춘다."""
    with Image.open(path) as im:
        if max_long and im.format in ("JPEG", "MPO"):
            w, h = im.size
            s = max_long / max(w, h)
            if s < 1.0:
                im.draft("RGB", (max(1, int(w * s)), max(1, int(h * s))))
        im = ImageOps.exif_transpose(im).convert("RGB")
        rgb = np.array(im)
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)


def resize_long(img, long_side):
    """긴 변을 long_side 로 축소. 확대는 하지 않는다. (결과, 축척) 반환."""
    h, w = img.shape[:2]
    s = long_side / max(h, w)
    if s >= 1.0:
        return img, 1.0
    return cv2.resize(img, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA), s


def crop_with_margin(img, bbox, margin):
    """bbox=(x0,y0,x1,y1) 원본 좌표. 박스 크기 비율만큼 여백을 두고 크롭."""
    H, W = img.shape[:2]
    x0, y0, x1, y1 = bbox
    mw, mh = (x1 - x0) * margin, (y1 - y0) * margin
    x0, x1 = int(max(0, x0 - mw)), int(min(W, x1 + mw))
    y0, y1 = int(max(0, y0 - mh)), int(min(H, y1 + mh))
    if x1 <= x0 or y1 <= y0:
        return None
    return img[y0:y1, x0:x1]

In [ ]:
# 날짜 후보 정규식. (?<!\d) / (?!\d) 로 더 긴 숫자열의 일부를 잘라 읽는 것을 막는다.
_SEP = r"\s*[.\-/]\s*"
_P_YYYY = re.compile(r"(?<!\d)(20\d{2})[.\-/\s]+(\d{1,2})[.\-/\s]+(\d{1,2})")   # 2027.02.14 / 2027 02 14 / '2027.02.1416:01'(공백 탈락) 허용
_P_CMP  = re.compile(r"(?<!\d)(20\d{2})(\d{2})(\d{2})(?!\d)")                          # 20270214 (점 탈락)
# 2자리 연도는 잡음과 혼동되기 쉬우므로 구분자 주변 공백 불허 + 같은 구분자 반복(역참조) 요구. '30.3- 7' 같은 조각 차단.
_P_YY   = re.compile(r"(?<!\d)(\d{2})([.\-/])(\d{1,2})\2(\d{1,2})(?!\d)")                # 21.04.24 / 25-12-02
_P_SP   = re.compile(r"(?<!\d)(\d{1,2})\s+(\d{1,2})\s+(\d{2,4})(?!\d)")                 # 30 12 23 (유럽식)
_P_DMY4 = re.compile(r"(?<!\d)(\d{1,2})[.\-/](\d{1,2})[.\-/](20\d{2})(?!\d)")            # 20/05/2026 — 연도가 맨 뒤면 일/월/년 (운영진 확정)
_P_CMP6 = re.compile(r"(?<!\d)(\d{2})(\d{2})(\d{2})(?!\d)")                                # 050926 — 'BEST BEFORE (DDMMYY)' 압축형 / 250626 (YYMMDD)
# 연도 없는 월.일 (예: 우유 '10.14 09:45'). 운영진 확정: NONE-10-14 로 출력.
# 앞뒤에 '.'+숫자가 붙으면(= 완전한 날짜의 일부) 제외. 2자리+2자리만 허용해 '4.9%' 류를 거른다.
_P_MMDD = re.compile(r"(?<![\d.])(\d{2})[.\-/](\d{2})(?![.\-/]?\d)")
# 일 없는 연.월 (예: 일본 '2027.7', '2026.01'). 운영진 확정: YYYY-MM-NONE. 뒤에 구분자+숫자가 오면 완전한 날짜의 일부이므로 제외.
_P_YYYYMM = re.compile(r"(?<!\d)(20\d{2})[.\-/](\d{1,2})(?![.\-/]?\d)")
# 유럽식 월.연도 (예: '12.2020 -13:28/1819011'). 앞에 구분자+숫자가 오면 DD.MM.YYYY 의 일부이므로 제외.
_P_MMYYYY = re.compile(r"(?<![\d.\-/])(\d{1,2})[.\-/](20\d{2})(?!\d)")
# 2자리 연도.월 (예: '25.11' → 2025-11-NONE). 첫 숫자가 13 이상이면 월일 수 없으므로 연.월 (정본 §5). MM.DD 와 같은 모양이라 값으로 구분.
_P_YYMM = re.compile(r"(?<![\d.\-/])(\d{2})[.\-/](\d{1,2})(?![.\-/]?\d)")
# 'NN/NN' 이 한 줄에 둘 이상 병기되고 뒤 숫자가 전부 연도 범위면 월/연도 (정본 §5. 미국 제품 'EXP:08/22 MFD:08/20' → 2022-08-NONE)
_P_NN_NN = re.compile(r"(?<![\d.\-/])(\d{1,2})/(\d{2})(?![.\-/]?\d)")
# 영문 월. 앞뒤 영문자를 막아 MAYONNAISE·SEPARATE 안의 조각을 거른다. 구분자는 . - / 공백 또는 없음(22JUN2021).
_MON = r"(?<![A-Z])(JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|SEPT|OCT|NOV|DEC)[A-Z]{0,6}(?![A-Z])"
_MSEP = r"[.\-/\s]*"
_P_MON_DY = re.compile(rf"{_MON}{_MSEP}(\d{{1,2}})[.\-/\s]+(\d{{2,4}})(?!\d)", re.I)      # FEB/26/21, DEC-28-2021, NOV 29 2021. 일·연 사이 구분자 필수(Oct/2021 → 20,21 분리 방지)
_P_D_MON_Y = re.compile(rf"(?<!\d)(\d{{1,2}}){_MSEP}{_MON}{_MSEP}(\d{{2,4}})(?!\d)", re.I)  # 04-Jul-21, 11/Oct/2021, 22 JUN 2021
_P_Y_MON_D = re.compile(rf"(?<!\d)(20\d{{2}}){_MSEP}{_MON}{_MSEP}(\d{{1,2}})(?!\d)", re.I)  # 2021 JUN 22
_P_MON_Y = re.compile(rf"{_MON}{_MSEP}(20\d{{2}})(?!\d)", re.I)                                # NOV 2021 → 2021-11-NONE (라벨 1,303장 중 5건)
_P_Y_MON = re.compile(rf"(?<!\d)(20\d{{2}}){_MSEP}{_MON}", re.I)                                # 2021 NOV
_MONTHS = {m: i + 1 for i, m in enumerate(["JAN", "FEB", "MAR", "APR", "MAY", "JUN", "JUL", "AUG", "SEP", "OCT", "NOV", "DEC"])}
_MONTHS["SEPT"] = 9
# 일먼저(DD/MM/YY) 힌트 키워드. 수입 상품 포장의 'BBD', 'BEST BEFORE', 'BBE', 'EXP', 'DD/MM/YY' 표기.
_P_DMY_HINT = re.compile(r"\bBBD\b|\bBBE\b|BEST\s*BEFORE|\bEXP\b|\bDD\s*[./-]?\s*MM\b|\bUSE\s*BY\b", re.I)

# 영문자 허용 시 숫자가 유사 글자로 읽히는 것을 되돌린다 (실측: '2021.10.20' → '2021.10.2Om4T', '2020.11.04' → '2020. J .11.04').
#  1) 구분자 사이에 홀로 낀 영문자는 잡음으로 보고 지운다.
#  2) 숫자 바로 뒤의 혼동 글자(2O → 20), 숫자 바로 앞의 혼동 글자(O6 → 06, 단 앞이 영문자면 제외: FEB 의 B 보호),
#     '숫자+구분자' 뒤의 혼동 글자 연속(03.Il → 03.11)을 숫자로 바꾼다. 안정될 때까지 반복.
#  영문 월 이름(OCT, DEC 등)은 먼저 떼어 두고 나머지 구간에만 적용해 22OCT2021 이 220CT2021 로 망가지지 않게 한다.
_CONF_MAP = {"O": "0", "o": "0", "Q": "0", "I": "1", "l": "1", "i": "1", "|": "1", "Z": "2", "z": "2", "S": "5", "s": "5", "B": "8"}
_CONF_CLS = "[OoQIliZzSsB|]"
_P_MONWORD = re.compile(r"JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC", re.I)
_P_STRAY = re.compile(r"(?<=[.\-/\s])[A-Za-z](?=[.\-/\s])")
_P_AFTER_DIGIT = re.compile(rf"(?<=\d){_CONF_CLS}")
_P_BEFORE_DIGIT = re.compile(rf"(?<![A-Za-z]){_CONF_CLS}(?=\d)")
_P_AFTER_SEP = re.compile(rf"(?<=\d[.\-/]){_CONF_CLS}+(?![A-Za-z])")
_conf = lambda m: "".join(_CONF_MAP[c] for c in m[0])


def _fix_segment(seg):
    seg = _P_STRAY.sub(" ", seg)
    prev = None
    while prev != seg:
        prev = seg
        seg = _P_AFTER_DIGIT.sub(_conf, seg)
        seg = _P_BEFORE_DIGIT.sub(_conf, seg)
        seg = _P_AFTER_SEP.sub(_conf, seg)
    return seg


_P_OCT = re.compile(r"(?<![A-Za-z0-9])0(?=ct(?![a-z])|CT(?![A-Z]))")   # PaddleOCR 실측: 'Oct' 를 '0ct' 로 읽음


def fix_confusions(s):
    s = _P_OCT.sub("O", s)
    out, pos = [], 0
    for m in _P_MONWORD.finditer(s):
        out += [_fix_segment(s[pos:m.start()]), m[0]]
        pos = m.end()
    out.append(_fix_segment(s[pos:]))
    return "".join(out)


def has_dmy_hint(text):
    return bool(_P_DMY_HINT.search(text))


def _valid(y, m, d):
    """y 또는 d 가 None 이면 그 필드는 검사하지 않는다 (연도 없는 MM.DD, 일 없는 YYYY.MM)."""
    return (y is None or YEAR_MIN <= y <= YEAR_MAX) and (1 <= m <= 12) and (d is None or 1 <= d <= 31)


def _year(c):
    """2자리/4자리 연도 문자열 → 4자리 int."""
    c = int(c)
    return c if c >= 100 else 2000 + c


def mask_long_digits(s):
    """품목보고번호(20130628332176)·바코드(8801052043838)처럼 앞자리가 날짜로 읽히는 긴 숫자열 제거."""
    return re.sub(r"\d{%d,}" % MASK_DIGITS_GE, " ", s)


def parse_dates(text, dmy_hint=False):
    """한 줄 텍스트 → [(y, m, d, kind)] 후보. kind 는 어떤 패턴으로 잡혔는지(진단용).
    dmy_hint=True 면 2자리 연도의 양방향 해석(YY.MM.DD vs DD.MM.YY)이 둘 다 가능할 때 일먼저를 택한다
    (포장에 BBD·BEST BEFORE·DD/MM/YY 문구가 있을 때. 기본은 국내 관행 YY.MM.DD).
    신뢰도 순 4단계로 보고, 상위 단계에서 후보가 나오면 하위 단계는 보지 않는다:
      1) 구분자 있는 완전한 날짜 (2027.02.14 / 20270214 / 21.04.24) + 영문 월 (FEB/26/21, 04-Jul-21)
      2) 공백 구분 (30 12 23) — 영양성분표 숫자열과 혼동되기 쉬워 1)이 없을 때만
      3) 일 없는 연.월 (2027.7 → YYYY-MM-NONE), 월.연도 (12.2020 → YYYY-MM-NONE)
      4) 연도 없는 월.일 (10.14) — 운영진 확정: NONE-MM-DD"""
    s = mask_long_digits(fix_confusions(text))
    s = re.sub(r"(?<!\d)(\d{2})(20\d{2})(?!\d)", r"\1 \2", s)   # 붙어 찍힌 'AUG292020' → 'AUG29 2020' (정본, label.py 와 동일)
    out = []
    for m in _P_MON_DY.finditer(s):
        mo, d, y = _MONTHS[m[1].upper()], int(m[2]), _year(m[3])
        if _valid(y, mo, d):
            out.append((y, mo, d, "MON_DY"))
    for m in _P_D_MON_Y.finditer(s):
        d, mo, y = int(m[1]), _MONTHS[m[2].upper()], _year(m[3])
        if _valid(y, mo, d):
            out.append((y, mo, d, "D_MON_Y"))
    for m in _P_Y_MON_D.finditer(s):
        y, mo, d = int(m[1]), _MONTHS[m[2].upper()], int(m[3])
        if _valid(y, mo, d):
            out.append((y, mo, d, "Y_MON_D"))
    for m in _P_YYYY.finditer(s):
        y, mo, d = int(m[1]), int(m[2]), int(m[3])
        if _valid(y, mo, d):
            out.append((y, mo, d, "YYYY"))
    for m in _P_CMP.finditer(s):
        y, mo, d = int(m[1]), int(m[2]), int(m[3])
        if _valid(y, mo, d):
            out.append((y, mo, d, "CMP"))
    for m in _P_DMY4.finditer(s):
        d, mo, y = int(m[1]), int(m[2]), int(m[3])
        if _valid(y, mo, d):
            out.append((y, mo, d, "DMY4"))
    for m in _P_YY.finditer(s):
        a, b, c = int(m[1]), int(m[3]), int(m[4])   # m[2] 는 구분자(역참조용)
        # 정본 §4: 년/월/일 우선. 일/월/년은 (1) 키워드 힌트, (2) 년/월/일이 무효, (3) 년/월/일 연도가 2028+ 이고 일/월/년 성립일 때.
        ymd, dmy = (2000 + a, b, c), (2000 + c, b, a)
        if _valid(*dmy) and (dmy_hint or not _valid(*ymd) or ymd[0] >= YY_FLIP_FROM):
            out.append((*dmy, "DDMMYY"))
        elif _valid(*ymd):
            out.append((*ymd, "YY"))
    if not out:
        for m in _P_SP.finditer(s):
            a, b, c = int(m[1]), int(m[2]), int(m[3])
            cy = c if c >= 100 else 2000 + c
            # 공백 구분은 유럽식 DD MM YY 우선 (예: 이탈리아 제품 '30 12 23'), 안 되면 YY MM DD
            if _valid(cy, b, a):
                out.append((cy, b, a, "SP_DMY"))
            elif c < 100 and _valid(2000 + a, b, c):
                out.append((2000 + a, b, c, "SP_YMD"))
        for m in _P_CMP6.finditer(s):
            a, b, c = int(m[1]), int(m[2]), int(m[3])
            # 정본 §3: YYMMDD 우선, 안 되면 DDMMYY (수입 과자 'BEST BEFORE (DDMMYY) 050926'). 2자리 연도 규칙(§4)과 같은 뒤집기 조건. LOT 번호는 연도 범위에서 대부분 탈락.
            ymd, dmy = (2000 + a, b, c), (2000 + c, b, a)
            if _valid(*dmy) and (dmy_hint or not _valid(*ymd) or ymd[0] >= YY_FLIP_FROM):
                out.append((*dmy, "CMP6_DMY"))
            elif _valid(*ymd):
                out.append((*ymd, "CMP6_YMD"))
    if not out:
        for m in _P_YYYYMM.finditer(s):
            y, mo = int(m[1]), int(m[2])
            if _valid(y, mo, None):
                out.append((y, mo, None, "YYYYMM"))
        for m in _P_MMYYYY.finditer(s):
            mo, y = int(m[1]), int(m[2])
            if _valid(y, mo, None):
                out.append((y, mo, None, "MMYYYY"))
        nn = [(int(m[1]), 2000 + int(m[2])) for m in _P_NN_NN.finditer(s)]
        if len(nn) >= 2 and all(_valid(y, mo, None) for mo, y in nn):
            for mo, y in nn:
                out.append((y, mo, None, "MM_YY"))
        for m in _P_YYMM.finditer(s):
            a, b = int(m[1]), int(m[2])
            if a >= 13 and _valid(2000 + a, b, None):
                out.append((2000 + a, b, None, "YYMM"))
        for m in _P_MON_Y.finditer(s):
            mo, y = _MONTHS[m[1].upper()], int(m[2])
            if _valid(y, mo, None):
                out.append((y, mo, None, "MON_Y"))
        for m in _P_Y_MON.finditer(s):
            y, mo = int(m[1]), _MONTHS[m[2].upper()]
            if _valid(y, mo, None):
                out.append((y, mo, None, "Y_MON"))
    if not out:
        for m in _P_MMDD.finditer(s):
            mo, d = int(m[1]), int(m[2])
            if _valid(None, mo, d):
                out.append((None, mo, d, "MMDD"))
    seen, uniq = set(), []
    for t in out:
        if t[:3] not in seen:
            seen.add(t[:3])
            uniq.append(t)
    return uniq


# 빠른 자가검증 (실행 시 assert 통과해야 함)
assert parse_dates("소비기한 2027.06.26 까지")[0][:3] == (2027, 6, 26)
assert parse_dates("품목보고번호 20130628332176")     == []            # 14자리 마스킹
assert parse_dates("21.04.24 까지 10:47")[0][:3]       == (2021, 4, 24)
assert parse_dates("30 12 23")[0][:3]                  == (2023, 12, 30) # DD MM YY
assert parse_dates("2020.12.28/16:35 2021.12.27(3)")   and len(parse_dates("2020.12.28/16:35 2021.12.27(3)")) == 2
assert parse_dates("NEL 1857 CANNELLA")                == []
assert parse_dates("10.14 09:45 PA")[0][:3]            == (None, 10, 14) # 연도 없음 → NONE-10-14
assert all(t[0] is not None for t in parse_dates("2020.12.28"))          # 완전한 날짜에서 월.일 오탐 금지
assert parse_dates("4.9 2450 100 130")                 == []
assert parse_dates("BBD: 20/05/2026 YYT Q5")[0][:3]    == (2026, 5, 20)  # 연도 뒤 → 일/월/년 (운영진 확정)
assert parse_dates("050926 213")[0][:3]                == (2026, 9, 5)   # DDMMYY 압축형: 05일 09월 26년 (운영진 확정)
assert parse_dates("LOT NO 543123")                    == []             # 6자리 LOT 은 연도 범위에서 탈락
assert parse_dates("5069351 930117 113809")            == []
assert parse_dates("17.12.20")[0][:3]                  == (2017, 12, 20)  # 2017 은 실존 라벨 → 연도 하한 2017
assert parse_dates("16.02.21")[0][:3]                  == (2021, 2, 16)   # 정본 §4-2: 2016 은 범위 밖 → 일/월/년
assert parse_dates("28/02/22")[0][:3]                  == (2022, 2, 28)   # 정본 §4-3: 2028+ 이고 일/월/년 성립 → 일/월/년
assert parse_dates("30/07/26")[0][:3]                  == (2026, 7, 30)
assert parse_dates("27.06.26")[0][:3]                  == (2027, 6, 26)   # 2027 은 년/월/일 유지
assert parse_dates("25.11")[0][:3]                     == (2025, 11, None) # 정본 §5: 첫 숫자 13+ → 연.월
assert parse_dates("EXP:08/22 MFD:08/20")[0][:3]       == (2022, 8, None)  # 정본 §5: NN/NN 병기 → 월/연도
assert parse_dates("AUG292020")[0][:3]                 == (2020, 8, 29)   # 붙어 찍힌 영문 월
assert parse_dates("BEST BEFORE NOV 2021")[0][:3]      == (2021, 11, None) # 영문 월 + 연도만 → 일 NONE
assert parse_dates("NOV 29 2021")                      == [(2021, 11, 29, "MON_DY")]  # 완전한 날짜가 있으면 MON_Y 는 안 봄
assert parse_dates("FEB/26/21")[0][:3]                 == (2021, 2, 26)   # 영문 월 MON/DD/YY
assert parse_dates("04-Jul-21")[0][:3]                 == (2021, 7, 4)    # DD-Mon-YY
assert parse_dates("11/Oct/2021")                      == [(2021, 10, 11, "D_MON_Y")]  # Oct/2021 을 MON DD YY 로 오탐 금지
assert parse_dates("22 JUN 2021")[0][:3]               == (2021, 6, 22)
assert parse_dates("DEC-28-2021")[0][:3]               == (2021, 12, 28)
assert parse_dates("BBE NOV 29 2021")[0][:3]           == (2021, 11, 29)
assert parse_dates("MAYONNAISE 2021")                  == []             # 단어 속 MAY 오탐 금지
assert parse_dates("12.2020 -13:28/1819011")[0][:3]    == (2020, 12, None) # 유럽식 월.연도 → YYYY-MM-NONE
assert parse_dates("07.2022")[0][:3]                   == (2022, 7, None)
assert parse_dates("2027.7")[0][:3]                    == (2027, 7, None)  # 일 없는 연.월
assert parse_dates("2026.01")[0][:3]                   == (2026, 1, None)
assert parse_dates("17.06.2026")                       == [(2026, 6, 17, "DMY4")]  # 06.2026 을 월.연도로 오탐 금지
assert parse_dates("2026.01.21")                       == [(2026, 1, 21, "YYYY")]  # 2026.01 을 연.월로 오탐 금지
assert parse_dates("24/12/21")[0][:3]                  == (2024, 12, 21)  # 기본은 YY.MM.DD (2024 < 2028)
assert parse_dates("24/12/21", dmy_hint=True)[0][:3]   == (2021, 12, 24)  # BBD 등 힌트가 있으면 DD/MM/YY
assert parse_dates("2O27.O6.26")[0][:3]                == (2027, 6, 26)   # O→0 혼동 복원
assert parse_dates("SEP 2021")[0][:3]                  == (2021, 9, None) # 영문 월 토큰의 S 는 5 로 바꾸지 않는다 (MON YYYY → 일 NONE)
assert parse_dates("2021.10.2Om4T 9k13")[0][:3]        == (2021, 10, 20)  # 실측 오독: 숫자 뒤 O → 0
assert parse_dates("2020. J .11.04 J 20:J8:")[0][:3]   == (2020, 11, 4)   # 실측 오독: 구분자 사이 홀로 낀 글자 제거
assert parse_dates("2026.03.Il")[0][:3]                == (2026, 3, 11)   # 구분자 뒤 혼동 글자 연속
assert parse_dates("22OCT2021")[0][:3]                 == (2021, 10, 22)  # 월 이름 안의 O 는 보호
assert parse_dates("FEB/26/21 OK")[0][:3]              == (2021, 2, 26)   # FEB 의 B 는 보호
assert parse_dates("Best before 11/0ct/2021")[0][:3]   == (2021, 10, 11)  # 0ct → Oct
assert has_dmy_hint("BEST BEFORE 24/12/21") and not has_dmy_hint("소비기한 2027.06.26")
print("parse_dates 자가검증 통과")

In [ ]:
easy_reader = None
if "craft" in (DET_BACKEND,) or REC_BACKEND == "easyocr":
    easy_reader = easyocr.Reader(
        ["en"], gpu=USE_GPU,
        model_storage_directory=WEIGHTS_DIR,
        download_enabled=False,   # 오프라인 강제: 가중치가 없으면 여기서 즉시 실패해야 한다
    )


def _need(model_dir, what):
    if not os.path.exists(os.path.join(model_dir, "inference.pdiparams")):
        raise FileNotFoundError(f"PaddleOCR {what} 가중치 없음: {model_dir}  (download_weights.sh 실행 필요)")


class PaddleReader:
    """easyocr.Reader 와 같은 detect()/recognize() 인터페이스. 탐지·인식을 각각 PaddleOCR 또는 EasyOCR 로 고를 수 있다.
    - torch 를 paddle 보다 먼저 import 해야 한다 (Windows 에서 DLL 충돌). 이 노트북은 위 셀에서 torch 를 먼저 올린다.
    - Paddle 이 predict 를 부르면 torch 의 스레드 수가 1로 떨어진다 (OpenMP 전역 설정 공유). CRAFT 를 쓸 때는 탐지 직전마다 되돌린다.
    - Paddle 은 allowlist 가 없으므로 인식 결과에서 허용 문자 밖의 글자를 공백으로 바꾼다 (삭제하면 '2021(3)' → '20213' 처럼 숫자가 붙는다).
    - recognize(img, horizontal_list) 는 박스들을 한 번의 predict 로 배치 인식하고 입력 순서를 유지한다."""

    def __init__(self, easy, det_backend, rec_backend, det_dir, rec_dir, threads):
        self.easy, self.det_backend, self.rec_backend = easy, det_backend, rec_backend
        if det_backend == "paddle" or rec_backend == "paddle":
            from paddleocr import TextDetection, TextRecognition
        if det_backend == "paddle":
            _need(det_dir, "탐지기")
            self.det = TextDetection(model_name=os.path.basename(det_dir), model_dir=det_dir, device="cpu", cpu_threads=threads, enable_mkldnn=False)
        if rec_backend == "paddle":
            _need(rec_dir, "인식기")
            self.rec = TextRecognition(model_name=os.path.basename(rec_dir), model_dir=rec_dir, device="cpu", cpu_threads=threads)

    # ---- 탐지: (horizontal_list, free_list) 를 EasyOCR 형식으로 돌려준다. horizontal_list[0] = [[x0, x1, y0, y1], ...]
    def detect(self, img, **k):
        if self.det_backend == "craft":
            torch.set_num_threads(TORCH_THREADS)
            return self.easy.detect(img, **k)
        H, W = img.shape[:2]
        boxes = []
        for poly in list(self.det.predict(input=img))[0]["dt_polys"]:
            xs, ys = [int(p[0]) for p in poly], [int(p[1]) for p in poly]
            x0, x1, y0, y1 = max(0, min(xs)), min(W, max(xs)), max(0, min(ys)), min(H, max(ys))
            if x1 - x0 >= 2 and y1 - y0 >= 2:
                boxes.append([x0, x1, y0, y1])
        return [boxes], [[]]

    # ---- 인식
    def _rec_many(self, crops, allowlist):
        """크롭 리스트 → [(text, conf)] 같은 순서. 빈 크롭은 ("", 0.0)."""
        idx, batch = [], []
        for i, c in enumerate(crops):
            if c is None or c.size == 0 or min(c.shape[:2]) < 2:
                continue
            batch.append(cv2.cvtColor(c, cv2.COLOR_GRAY2BGR) if c.ndim == 2 else c)
            idx.append(i)
        out = [("", 0.0)] * len(crops)
        if not batch:
            return out
        if self.rec_backend == "paddle":
            for i, r in zip(idx, self.rec.predict(input=batch, batch_size=len(batch))):
                text = r["rec_text"]
                if allowlist:
                    text = "".join(ch if ch in allowlist else " " for ch in text)
                out[i] = (text, float(r["rec_score"]))
        else:
            for i, c in zip(idx, batch):
                res = self.easy.recognize(c, allowlist=allowlist)
                out[i] = (res[0][1], float(res[0][2])) if res else ("", 0.0)
        return out

    def recognize(self, img, horizontal_list=None, free_list=None, allowlist=None, reformat=True, **k):
        if horizontal_list:
            crops = [img[y0:y1, x0:x1] for (x0, x1, y0, y1) in horizontal_list]
            return [([[x0, y0], [x1, y0], [x1, y1], [x0, y1]], t, c)
                    for (x0, x1, y0, y1), (t, c) in zip(horizontal_list, self._rec_many(crops, allowlist))]
        h, w = img.shape[:2]
        (t, c), = self._rec_many([img], allowlist)
        return [([[0, 0], [w, 0], [w, h], [0, h]], t, c)]


if DET_BACKEND == "craft" and REC_BACKEND == "easyocr":
    reader = easy_reader
else:
    reader = PaddleReader(easy_reader, DET_BACKEND, REC_BACKEND, PADDLE_DET_DIR, PADDLE_REC_DIR, PADDLE_THREADS)
print(f"로드 완료 (offline): 탐지 {'PaddleOCR ' + os.path.basename(PADDLE_DET_DIR) if DET_BACKEND == 'paddle' else 'EasyOCR CRAFT'}"
      f" + 인식 {'PaddleOCR ' + os.path.basename(PADDLE_REC_DIR) if REC_BACKEND == 'paddle' else 'EasyOCR english_g2'}")

In [ ]:
def group_lines(results, y_tol=0.6):
    """readtext 결과를 같은 줄끼리 묶어 [(text, (x0,y0,x1,y1), conf)] 반환.
    '30 12 23' 처럼 공백으로 갈라진 날짜, '2020.12.28/16:35' 처럼 붙은 시간을 한 줄로 본다."""
    items = []
    for box, text, conf in results:
        xs = [p[0] for p in box]
        ys = [p[1] for p in box]
        items.append({"x0": min(xs), "y0": min(ys), "x1": max(xs), "y1": max(ys),
                      "text": text, "conf": float(conf)})
    items.sort(key=lambda t: ((t["y0"] + t["y1"]) / 2, t["x0"]))
    lines = []
    for it in items:
        cy, h = (it["y0"] + it["y1"]) / 2, it["y1"] - it["y0"]
        for ln in lines:
            lcy, lh = (ln["y0"] + ln["y1"]) / 2, ln["y1"] - ln["y0"]
            if abs(cy - lcy) <= max(h, lh) * y_tol:
                ln["items"].append(it)
                ln["x0"], ln["x1"] = min(ln["x0"], it["x0"]), max(ln["x1"], it["x1"])
                ln["y0"], ln["y1"] = min(ln["y0"], it["y0"]), max(ln["y1"], it["y1"])
                break
        else:
            lines.append({"x0": it["x0"], "y0": it["y0"], "x1": it["x1"], "y1": it["y1"], "items": [it]})
    for ln in lines:
        ln["items"].sort(key=lambda t: t["x0"])
        ln["text"] = " ".join(t["text"] for t in ln["items"])
        ln["conf"] = float(np.mean([t["conf"] for t in ln["items"]]))
    return lines   # 각 원소: {x0,y0,x1,y1,text,conf,items:[단어 박스…]}


def ocr_prioritized(small):
    """검출 1회 → 박스를 글자 높이 내림차순으로 인식(조기 종료·상한 적용). readtext 와 같은 [(box, text, conf)] 반환.
    기울어진 폴리곤(free_list)은 축 정렬 bbox 로 바꿔 같이 다룬다."""
    hl, fl = reader.detect(small, min_size=10)
    boxes = [tuple(map(int, b)) for b in hl[0]]                         # (x0, x1, y0, y1)
    for poly in fl[0]:
        xs, ys = [p[0] for p in poly], [p[1] for p in poly]
        boxes.append((int(min(xs)), int(max(xs)), int(min(ys)), int(max(ys))))
    boxes.sort(key=lambda b: -(b[3] - b[2]))                              # 큰 글자 먼저
    boxes = boxes[:MAX_BOXES]
    grey = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    out, stop_h, i = [], None, 0
    while i < len(boxes):
        chunk = boxes[i:i + REC_BATCH]                                     # REC_BATCH 개씩 배치 인식 (Paddle). 조기 종료는 배치 단위로.
        i += REC_BATCH
        if stop_h is not None and (chunk[0][3] - chunk[0][2]) < stop_h:
            break
        res = reader.recognize(grey, [list(b) for b in chunk], [], allowlist=ALLOW_P1, reformat=False)
        for (x0, x1, y0, y1), (_, text, conf) in zip(chunk, res):
            h = y1 - y0
            if stop_h is not None and h < stop_h:
                break
            out.append(([[x0, y0], [x1, y0], [x1, y1], [x0, y1]], text, float(conf)))
            if stop_h is None and any(p[0] is not None for p in parse_dates(text)):
                stop_h = h * EARLY_STOP_RATIO
    return out


def pass1(img):
    """해상도 사다리를 오르며 축소본 전체 OCR. 후보가 나오는 첫 단계에서 멈춘다.
    반환: (후보 리스트, [(해상도, OCR 라인들)] 진단용). bbox 는 원본 좌표로 환산해 둔다."""
    seen = []
    for L in PASS1_LADDER:
        small, s = resize_long(img, L)
        lines = group_lines(ocr_prioritized(small))
        seen.append((L, lines))
        hint = any(has_dmy_hint(ln["text"]) for ln in lines)   # 이미지 어디든 BBD·BEST BEFORE 가 있으면 일먼저
        cands = []
        for ln in lines:
            for (y, m, d, kind) in parse_dates(ln["text"], dmy_hint=hint):
                cands.append({"y": y, "m": m, "d": d, "kind": kind, "conf": ln["conf"], "src": f"p1@{L}", "text": ln["text"], "hint": hint,
                              "bbox":  (ln["x0"] / s, ln["y0"] / s, ln["x1"] / s, ln["y1"] / s),
                              "items": [(it["x0"] / s, it["y0"] / s, it["x1"] / s, it["y1"] / s) for it in ln["items"]]})
        if cands:
            return cands, seen
        if s >= 1.0:          # 이미 원본 크기 → 더 올려도 같은 그림
            break
    return [], seen


def pass2(img, cands):
    """후보 영역만 원본에서 크롭해 **인식기만** 다시 돌린다 (검출기는 장당 1회로 제한 — CPU 비용의 대부분이 검출기).
    크롭 전체를 한 줄로 보고 recognize() 하므로 검출 없이 수십 ms 에 끝난다.
    재인식이 파싱되면 그 결과로 대체, 아니면 1패스 값 유지.
    크롭에 이웃 줄(제조일자 등)이 같이 들어와 날짜가 여러 개 나오면 전부 후보로 올린다 → 선택 규칙이 처리."""
    out = []
    for c in cands:
        # 줄 전체가 아니라 검출된 단어 박스 단위로 재인식 → 이웃 줄이 섞여 들어오는 것 방지. 크롭들은 한 번에 배치 인식.
        crops = [cr for cr in (crop_with_margin(img, ib, PASS2_MARGIN) for ib in c["items"]) if cr is not None]
        if hasattr(reader, "_rec_many"):
            pairs = reader._rec_many(crops, ALLOW_P2)
        else:
            pairs = [(r[0][1], float(r[0][2])) if (r := reader.recognize(cr, allowlist=ALLOW_P2)) else ("", 0.0) for cr in crops]
        texts = [t for t, _ in pairs]
        confs = [cf for _, cf in pairs]
        text = " ".join(texts)
        conf = float(np.mean(confs)) if confs else 0.0
        parsed = parse_dates(text, dmy_hint=c.get("hint", False))
        if c["y"] is not None:
            # 1패스가 연도까지 읽었는데 2패스가 연도를 잃으면(크롭 경계 잘림) 1패스를 신뢰
            parsed = [p for p in parsed if p[0] is not None]
        if parsed:
            for (y, m, d, kind) in parsed:
                out.append({**c, "y": y, "m": m, "d": d, "kind": kind, "conf": conf, "src": "p2", "text": text})
        else:
            out.append(c)
    return out


def select_date(cands):
    """운영진 확정 규칙: 날짜가 여럿이면 가장 늦은 것이 소비기한. 동률은 신뢰도."""
    if not cands:
        return None
    # 연도 없는 후보(y=None)·일 없는 후보(d=None)는 같은 연·월의 완전한 날짜보다 뒤로 보낸다
    return max(cands, key=lambda c: ((c["y"] or 0, c["m"], c["d"] or 0), c["conf"]))

In [ ]:
all_files   = sorted(glob.glob(os.path.join(INPUT_DIR, "*.*")))
image_files = [p for p in all_files if os.path.splitext(p)[1].lower() in IMG_EXT]
skipped     = [os.path.basename(p) for p in all_files if p not in set(image_files)]
if skipped:
    print(f"[INFO] 이미지 확장자가 아니어서 건너뜀: {skipped[:10]}{' ...' if len(skipped) > 10 else ''}")
print(f"입력 {len(image_files)}장  ({INPUT_DIR})")

rows, kinds = [], Counter()
n_fallback = n_p2 = n_err = 0
t_all = time.time()

for i, path in enumerate(image_files, 1):
    img_id = os.path.splitext(os.path.basename(path))[0]   # 확장자 제외 파일명 그대로. zero-pad 가정 금지.
    best, cands, p1_lines, t0 = None, [], [], time.time()
    try:
        img = load_image(path)
        cands, p1_lines = pass1(img)
        if cands:
            cands = pass2(img, cands)
        best = select_date(cands)
    except Exception as e:
        n_err += 1
        print(f"[WARN] {img_id}: {type(e).__name__}: {e}")
    if DEBUG:
        def fmt(c):
            return f"{c['y'] or 'NONE'}-{c['m']:02d}-" + ("NONE" if c["d"] is None else f"{c['d']:02d}")
        picked = f"{fmt(best)} ({best['kind']}/{best['src']})" if best else "NONE"
        print(f"  {img_id}: {time.time() - t0:.1f}s → {picked}")
        for c in cands:
            print(f"      cand {fmt(c)} {c['kind']:8s} {c['src']:7s} conf={c['conf']:.2f}  text='{c['text'][:80]}'")
        if not cands:
            for L, lines in p1_lines:   # 후보가 없을 때 OCR 이 실제로 뭘 읽었는지 (숫자 2개 이상 포함 라인만)
                digs = [(ln["text"][:40], round(ln["conf"], 2)) for ln in lines if re.search(r"\d{2}", ln["text"])]
                print(f"      p1@{L} 숫자 라인 {len(digs)}개: {digs[:10]}")

    if best is None:
        n_fallback += 1
        y, m, d = PRIOR_DATE if NONE_POLICY == "prior" else (None, None, None)
    else:
        y, m, d = best["y"], best["m"], best["d"]
        kinds[best["kind"]] += 1
        if best["src"] == "p2":
            n_p2 += 1

    # 각 필드는 독립적으로 NONE 가능. final_date 는 세 필드를 '-' 로 이은 것 (예: NONE-10-14, 운영진 확정 포맷).
    ys = f"{y:04d}" if y is not None else "NONE"
    ms = f"{m:02d}" if m is not None else "NONE"
    ds = f"{d:02d}" if d is not None else "NONE"
    fd = "NONE" if (y is None and m is None and d is None) else f"{ys}-{ms}-{ds}"
    rows.append({"image_id": img_id, "year": ys, "month": ms, "day": ds, "final_date": fd})

    if i % 50 == 0 or i == len(image_files):
        el = time.time() - t_all
        print(f"[{i}/{len(image_files)}] {el:.0f}s 경과 · 장당 {el / i:.2f}s · 후보없음 {n_fallback} · 오류 {n_err}")

In [ ]:
df = pd.DataFrame(rows, columns=["image_id", "year", "month", "day", "final_date"])
df.to_csv(OUTPUT_PATH, index=False)

total = time.time() - t_all
print(f"Saved {len(df)} rows → {OUTPUT_PATH}")
print(f"총 {total:.1f}s · 장당 {total / max(1, len(df)):.2f}s · 후보없음 {n_fallback} · 2패스 채택 {n_p2} · 오류 {n_err}")
print("패턴 분포:", dict(kinds))
df.head(10)